# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hadiya27/hadiya-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [26]:
import pandas as pd
import duckdb

In [27]:
# Connect to DuckDB
con = duckdb.connect()

print("DuckDB connection ready.")

DuckDB connection ready.


In [28]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

print("HF_TOKEN available:", hf_token is not None)

HF_TOKEN available: True


In [29]:
from google.colab import userdata
import duckdb

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

print("FlyRank warehouse connection setup complete.")

FlyRank warehouse connection setup complete.


In [30]:
result = con.execute("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").fetchdf()

result


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


In [31]:
# Inspect the March warehouse columns we can use for the baseline
columns = con.execute("""
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").fetchdf()

columns[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 1. My rule and its reason codes

* My Rules

I will use a simple decision-support score based on two observed signals: content staleness and CTR relative to average position. Older content receives more priority for review, while content with lower CTR at weaker average positions receives more priority. The score is used to rank pages for human review, not to predict Google's algorithm.

### Signal checks

*1. Staleness — CONFIRMED (directional).*
In the March 2026 snapshot, average impressions decreased across the observed staleness buckets: 84.8 for content updated within 90 days, 57.8 for 90–179 days, and 9.3 for 180–364 days. The older buckets are small, so I treat this as directional evidence rather than proof.

*2. CTR vs. position — CONFIRMED.*
Average CTR decreased consistently as average position became weaker: 0.371% for positions 1–3, 0.328% for 4–5, 0.272% for 6–10, and 0.162% for 11+. This supports using the signal as a decision-support input.

### Reason codes

- `STALE_CONTENT` — content is older and receives priority for review.
- `LOW_CTR_POSITION` — CTR is relatively low within a weaker position bucket.*

In [32]:
# Signal 1: content staleness
# Use the latest March observation for each content item.

staleness_check = con.execute("""
WITH latest_march AS (
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE report_date = DATE '2026-03-31'
),
joined AS (
    SELECT
        f.content_hash_id,
        f.gsc_impressions,
        f.gsc_clicks,
        f.gsc_avg_position,
        d.content_updated_date,
        DATE '2026-03-31' - d.content_updated_date AS days_since_update
    FROM latest_march f
    LEFT JOIN read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    ) d
    ON f.content_hash_id = d.content_hash_id
    WHERE f.gsc_data_available = TRUE
)
SELECT
    CASE
        WHEN days_since_update < 90 THEN '<90 days'
        WHEN days_since_update < 180 THEN '90-179 days'
        WHEN days_since_update < 365 THEN '180-364 days'
        ELSE '365+ days'
    END AS staleness_bucket,
    COUNT(*) AS n,
    ROUND(AVG(gsc_impressions), 1) AS avg_impressions
FROM joined
WHERE days_since_update IS NOT NULL
GROUP BY 1
ORDER BY
    CASE staleness_bucket
        WHEN '<90 days' THEN 1
        WHEN '90-179 days' THEN 2
        WHEN '180-364 days' THEN 3
        ELSE 4
    END
""").fetchdf()

staleness_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,avg_impressions
0,<90 days,124861,84.8
1,90-179 days,305,57.8
2,180-364 days,56,9.3


In [33]:
# Signal 2: CTR relative to average position

ctr_position_check = con.execute("""
WITH latest_march AS (
    SELECT
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE report_date = DATE '2026-03-31'
      AND gsc_data_available = TRUE
      AND gsc_impressions > 0
      AND gsc_avg_position > 0
),
scored AS (
    SELECT
        *,
        100.0 * gsc_clicks / gsc_impressions AS ctr_pct
    FROM latest_march
)
SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN '1-3'
        WHEN gsc_avg_position <= 5 THEN '4-5'
        WHEN gsc_avg_position <= 10 THEN '6-10'
        ELSE '11+'
    END AS position_bucket,
    COUNT(*) AS n,
    ROUND(AVG(ctr_pct), 3) AS avg_ctr_pct
FROM scored
GROUP BY 1
ORDER BY
    CASE position_bucket
        WHEN '1-3' THEN 1
        WHEN '4-5' THEN 2
        WHEN '6-10' THEN 3
        ELSE 4
    END
""").fetchdf()

ctr_position_check

,position_bucket,n,avg_ctr_pct
0,1-3,19913,0.371
1,4-5,20046,0.328
2,6-10,31960,0.272
3,11+,47088,0.162


## 2. Build the ranked queue (writes the CSV)

*### Rule implementation

*I use a transparent hand-written score. Stale content and weak CTR relative to position each add points to the review priority. The highest-scoring pages are ranked first. Each page receives one reason code and one action label so the queue can support a human review process.*

In [34]:
# Section 2: Build the baseline ranked queue

queue = con.execute("""
WITH latest_march AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE report_date = DATE '2026-03-31'
      AND gsc_data_available = TRUE
),
joined AS (
    SELECT
        f.*,
        d.content_updated_date,
        DATE '2026-03-31' - d.content_updated_date AS days_since_update,
        CASE
            WHEN f.gsc_impressions > 0
            THEN 100.0 * f.gsc_clicks / f.gsc_impressions
            ELSE NULL
        END AS ctr_pct
    FROM latest_march f
    LEFT JOIN read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    ) d
    ON f.content_hash_id = d.content_hash_id
),
scored AS (
    SELECT
        *,
        (
            CASE
                WHEN days_since_update >= 180 THEN 2
                WHEN days_since_update >= 90 THEN 1
                ELSE 0
            END
            +
            CASE
                WHEN gsc_avg_position > 10 AND ctr_pct < 0.20 THEN 2
                WHEN gsc_avg_position > 5 AND ctr_pct < 0.30 THEN 1
                ELSE 0
            END
        ) AS action_score
    FROM joined
    WHERE days_since_update IS NOT NULL
      AND gsc_avg_position > 0
      AND ctr_pct IS NOT NULL
)
SELECT
    content_hash_id,
    client_hash_id,
    report_date,
    action_score,
    CASE
        WHEN days_since_update >= 180
             AND gsc_avg_position > 10
             AND ctr_pct < 0.20
            THEN 'STALE_CONTENT'
        WHEN gsc_avg_position > 10
             AND ctr_pct < 0.20
            THEN 'LOW_CTR_POSITION'
        WHEN days_since_update >= 180
            THEN 'STALE_CONTENT'
        WHEN gsc_avg_position > 5
             AND ctr_pct < 0.30
            THEN 'LOW_CTR_POSITION'
        ELSE 'REVIEW'
    END AS reason_code,
    CASE
        WHEN action_score >= 3 THEN 'PRIORITIZE_REVIEW'
        WHEN action_score >= 2 THEN 'REVIEW'
        ELSE 'MONITOR'
    END AS action,
    days_since_update,
    gsc_avg_position,
    ctr_pct
FROM scored
ORDER BY action_score DESC, content_hash_id
""").fetchdf()

queue.head(20)


,content_hash_id,client_hash_id,report_date,action_score,reason_code,action,days_since_update,gsc_avg_position,ctr_pct
0,content_19f71daba0876547,client_65de48885f4ef01b,2026-03-31,4,STALE_CONTENT,PRIORITIZE_REVIEW,303,48.000000,0.000000
1,content_2879353f031b1983,client_65de48885f4ef01b,2026-03-31,4,STALE_CONTENT,PRIORITIZE_REVIEW,232,12.500000,0.000000
2,content_8af31b9bc99a03fd,client_73cda7b4e4f265ea,2026-03-31,4,STALE_CONTENT,PRIORITIZE_REVIEW,235,96.000000,0.000000
3,content_8d1bce995359d3b8,client_2b4306c3ed003f01,2026-03-31,4,STALE_CONTENT,PRIORITIZE_REVIEW,191,34.000000,0.000000
4,content_8fdcc6b2f67934ff,client_2b4306c3ed003f01,2026-03-31,4,STALE_CONTENT,PRIORITIZE_REVIEW,191,20.000000,0.000000
5,content_b9decfa128f0a643,client_65de48885f4ef01b,2026-03-31,4,STALE_CONTENT,PRIORITIZE_REVIEW,232,29.000000,0.000000
6,content_ebd0ea1607cd3ffc,client_73cda7b4e4f265ea,2026-03-31,4,STALE_CONTENT,PRIORITIZE_REVIEW,243,90.000000,0.000000
7,content_f406ec8774d96559,client_b10cb2997d0c7c86,2026-03-31,4,STALE_CONTENT,PRIORITIZE_REVIEW,234,16.600000,0.000000
8,content_f4098d5b2c2eeb18,client_73cda7b4e4f265ea,2026-03-31,4,STALE_CONTENT,PRIORITIZE_REVIEW,243,81.000000,0.000000
9,content_00adc350e8581721,client_08a6a72ff48e62c0,2026-03-31,3,LOW_CTR_POSITION,PRIORITIZE_REVIEW,131,66.000000,0.000000


In [35]:
import os

# Create the output folder if it does not already exist
os.makedirs("work/outputs", exist_ok=True)

# Save the ranked queue
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print(f"Saved ranked queue to: {output_path}")
print(f"Rows written: {len(queue):,}")

Saved ranked queue to: work/outputs/baseline_action_score.csv
Rows written: 119,007


## 3. Top-20 review

*I reviewed the highest-ranked 20 rows from the baseline queue. For each row, I record the action, reason code, a confidence note, and what could make the recommendation wrong. These are decision-support observations, not proof that a page needs a particular intervention.*

In [36]:
# Section 3: Top-20 review

top20 = queue.head(20).copy()

top20["confidence_note"] = top20.apply(
    lambda row:
        "Higher confidence because both staleness and CTR-position signals contribute."
        if row["action_score"] >= 3
        else "Directional signal; human review is still needed.",
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    lambda row:
        "The content may already be performing adequately despite the observed signals."
        if row["reason_code"] == "STALE_CONTENT"
        else "Low CTR may reflect limited impressions, query mix, or position rather than a content issue.",
    axis=1
)

top20[
    [
        "content_hash_id",
        "action",
        "reason_code",
        "action_score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]


,content_hash_id,action,reason_code,action_score,confidence_note,what_would_make_it_wrong
0,content_19f71daba0876547,PRIORITIZE_REVIEW,STALE_CONTENT,4,Higher confidence because both staleness and C...,The content may already be performing adequate...
1,content_2879353f031b1983,PRIORITIZE_REVIEW,STALE_CONTENT,4,Higher confidence because both staleness and C...,The content may already be performing adequate...
2,content_8af31b9bc99a03fd,PRIORITIZE_REVIEW,STALE_CONTENT,4,Higher confidence because both staleness and C...,The content may already be performing adequate...
3,content_8d1bce995359d3b8,PRIORITIZE_REVIEW,STALE_CONTENT,4,Higher confidence because both staleness and C...,The content may already be performing adequate...
4,content_8fdcc6b2f67934ff,PRIORITIZE_REVIEW,STALE_CONTENT,4,Higher confidence because both staleness and C...,The content may already be performing adequate...
5,content_b9decfa128f0a643,PRIORITIZE_REVIEW,STALE_CONTENT,4,Higher confidence because both staleness and C...,The content may already be performing adequate...
6,content_ebd0ea1607cd3ffc,PRIORITIZE_REVIEW,STALE_CONTENT,4,Higher confidence because both staleness and C...,The content may already be performing adequate...
7,content_f406ec8774d96559,PRIORITIZE_REVIEW,STALE_CONTENT,4,Higher confidence because both staleness and C...,The content may already be performing adequate...
8,content_f4098d5b2c2eeb18,PRIORITIZE_REVIEW,STALE_CONTENT,4,Higher confidence because both staleness and C...,The content may already be performing adequate...
9,content_00adc350e8581721,PRIORITIZE_REVIEW,LOW_CTR_POSITION,3,Higher confidence because both staleness and C...,"Low CTR may reflect limited impressions, query..."


## 4. Weak picks + leakage check

*Some picks may be weak because a low CTR can reflect limited impressions, query mix, or the page's position rather than a content problem. Staleness can also be a weak signal when content remains useful despite being old.

The queue uses March 31 observed fields only. I did not use future-window outcomes, product flags, trend labels, or other label-derived inputs. The score is intended for decision support and human review*

In [37]:
# Section 4: Weak picks + leakage check

print("Rows in ranked queue:", len(queue))
print("Top score:", queue["action_score"].max())
print("Lowest score:", queue["action_score"].min())

print("\nReason codes:")
print(queue["reason_code"].value_counts())

print("\nActions:")
print(queue["action"].value_counts())

# Check that no label/future-window fields are present
forbidden_terms = [
    "trend",
    "label",
    "declining",
    "future",
    "product_flag"
]

queue_columns_lower = [c.lower() for c in queue.columns]

leakage_columns = [
    c for c in queue_columns_lower
    if any(term in c for term in forbidden_terms)
]

print("\nPotential leakage columns found:", leakage_columns)

assert len(leakage_columns) == 0, "Potential label/future-window field found!"

print("Leakage check: PASS")


Rows in ranked queue: 119007
Top score: 4
Lowest score: 0

Reason codes:
reason_code
LOW_CTR_POSITION    73207
REVIEW              45753
STALE_CONTENT          47
Name: count, dtype: int64

Actions:
action
MONITOR              74861
REVIEW               43992
PRIORITIZE_REVIEW      154
Name: count, dtype: int64

Potential leakage columns found: []
Leakage check: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.